# Pramāṇa — Colab training notebook

Free T4 trains MobileNet-V3-Small on FF++ + Celeb-DF in ~3–4 hours.

**Setup:**
1. Runtime → Change runtime type → GPU.
2. Mount Google Drive (datasets and checkpoints live there).
3. Upload your FF++ + Celeb-DF extracted frames to `/content/drive/MyDrive/pramana_datasets/`.
4. Run all cells.

Bible §6 targets: AUC > 0.85 on FF++ val, > 0.75 on Celeb-DF val.

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo + install deps
!git clone https://github.com/TheClazer/project-pramana.git
%cd project-pramana
!pip install -q torch torchvision timm scikit-learn tqdm pillow opencv-python einops

In [ ]:
# Symlink Drive datasets into repo expected location
import os
os.makedirs('ml/datasets', exist_ok=True)
for name in ['faceforensics', 'celebdf', 'iiitcfw']:
    src = f'/content/drive/MyDrive/pramana_datasets/{name}'
    dst = f'ml/datasets/{name}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
!ls -la ml/datasets/

In [ ]:
# Quick smoke-test on stand-in first (verifies wiring without 30 GB FF++)
!python -m ml.src.train --dataset stand_in --epochs 1 --quick --out-dir /content/drive/MyDrive/pramana_runs/smoke

In [ ]:
# Full training run (MobileNet-V3-Small primary)
!python -m ml.src.train --dataset combined --epochs 25 --backbone mobilenet_v3_small \
    --batch-size 64 --lr 3e-4 \
    --out-dir /content/drive/MyDrive/pramana_runs/mobilenet_v3_small_v1

In [ ]:
# Eval the best checkpoint
!python -m ml.src.eval --checkpoint /content/drive/MyDrive/pramana_runs/mobilenet_v3_small_v1/best.pt --split val

## Next steps

1. Download `best.pt` to your laptop.
2. Run AI Hub Workbench compile + quantize + profile (locally, with your token):
   ```
   python -m ml.src.workbench.compile  --checkpoint best.pt --backbone mobilenet_v3_small
   python -m ml.src.workbench.quantize --compile-job-id <id> --precision int8
   python -m ml.src.workbench.profile  --quantize-job-id <id>
   ```
3. Drop the resulting `pramana-int8.tflite` into `android/app/src/main/assets/`.
